# 【notebook｜N-23】PyTorch 进阶与生态
> 对应：《00-预备/PY6-PyTorch进阶与生态.md》【文档｜DOC-PY6】；代码 `F-pre-py6`

> 目标：把检查点、非连续张量、混合精度、profiler/compile、以及
> transformers/peft/trl/accelerate 的接口跑一遍（CPU 可跑，可选依赖缺失即跳过）。


In [ ]:
# 【N-23.c1】
import importlib.util

def has(pkg):
    return importlib.util.find_spec(pkg) is not None

try:
    import torch
    print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
except ImportError:
    torch = None
    print('未安装 torch：pip install torch')
for p in ['transformers', 'peft', 'trl', 'accelerate']:
    print(f'{p:14s}', 'installed' if has(p) else 'missing')


## 实验 1：梯度检查点（用计算换显存）

【定义 PY6.1.1】：前向只存边界激活，反向时从最近检查点重算该段。
这里验证「检查点前后向」与「普通前后向」的输出和梯度一致。


In [ ]:
# 【N-23.c2】
if torch is None:
    print('跳过：需要 torch')
else:
    from torch import nn
    from torch.utils.checkpoint import checkpoint
    torch.manual_seed(0)
    seg = nn.Sequential(nn.Linear(16, 32), nn.ReLU(),
                        nn.Linear(32, 8), nn.ReLU(), nn.Linear(8, 1))
    run = lambda t: seg(t)
    x = torch.randn(8, 16)
    plain = run(x).detach()
    ck = checkpoint(run, x, use_reentrant=False).detach()
    print('前向一致:', bool(torch.allclose(plain, ck)))
    x1 = x.clone().requires_grad_(True)
    run(x1).sum().backward()
    x2 = x.clone().requires_grad_(True)
    checkpoint(run, x2, use_reentrant=False).sum().backward()
    print('梯度一致:', bool(torch.allclose(x1.grad, x2.grad, atol=1e-6)))


## 实验 2：非连续张量与 `view`/`reshape`

【注 PY6.1.2】：`transpose`/`permute` 产生非连续视图，`view` 会报错，
`reshape` 会自动兜底（必要时拷贝）。


In [ ]:
# 【N-23.c3】
if torch is None:
    print('跳过：需要 torch')
else:
    x = torch.arange(24).reshape(2, 3, 4)
    xt = x.transpose(1, 2)
    print('transpose 后 is_contiguous:', xt.is_contiguous())
    try:
        xt.view(2, 12)
    except RuntimeError as e:
        print('view 报错:', str(e).splitlines()[0])
    print('reshape :', tuple(xt.reshape(2, 12).shape))
    print('contig  :', tuple(xt.contiguous().view(2, 12).shape))


## 实验 3：混合精度 `autocast`（CPU 用 bf16 演示）

【定义 PY6.1.3】：`autocast` 只在作用域内改变算子计算 dtype，不改参数 dtype；
【注 PY6.1.4】：fp16 需 `GradScaler`，bf16 一般不需要。


In [ ]:
# 【N-23.c4】
if torch is None:
    print('跳过：需要 torch')
else:
    from torch import nn
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 1)).to(device)
    x = torch.randn(64, 32, device=device)
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        out = model(x)
    print('device:', device, '| 输出 dtype:', out.dtype)
    print('参数 dtype 仍为:', next(model.parameters()).dtype)
    if device != 'cuda':
        print('CPU 无 GradScaler 下溢问题')


## 实验 4：`torch.profiler` 找算子

【定义 PY6.2.1】：采集算子级事件，按耗时排序。


In [ ]:
# 【N-23.c5】
if torch is None:
    print('跳过：需要 torch')
else:
    from torch import nn
    model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64))
    x = torch.randn(256, 64)
    with torch.profiler.profile(
            activities=[torch.profiler.ProfilerActivity.CPU]) as prof:
        for _ in range(5):
            model(x).sum().backward()
            model.zero_grad(set_to_none=True)
    print(prof.key_averages().table(sort_by='cpu_time_total', row_limit=5))


## 实验 5：`torch.compile`（CPU 用 eager 后端避免长编译）

【定义 PY6.2.4】：形状固定时只编译一次；形状变化会触发重编译。


In [ ]:
# 【N-23.c6】
if torch is None:
    print('跳过：需要 torch')
elif not hasattr(torch, 'compile'):
    print('当前 torch 无 torch.compile')
else:
    from torch import nn
    model = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
    x = torch.randn(16, 32)
    backend = 'eager' if not torch.cuda.is_available() else 'inductor'
    compiled = torch.compile(model, backend=backend)
    print('backend:', backend, '| 前向:', tuple(compiled(x).shape))


## 实验 6：生态库速览（缺失即跳过，不下载权重）

【定义 PY6.3.1】–【定义 PY6.3.5】：`transformers` 加载、`peft` 省参数、
`trl` 管后训练、`accelerate` 管分布式。


In [ ]:
# 【N-23.c7】
if has('peft'):
    from peft import LoraConfig
    cfg = LoraConfig(r=8, lora_alpha=16, bias='none', task_type='CAUSAL_LM',
                     target_modules=['q_proj', 'v_proj'])
    print('LoraConfig ok: r=%d alpha=%d' % (cfg.r, cfg.lora_alpha))
else:
    print('peft 未安装：pip install peft')
if has('accelerate'):
    from accelerate import Accelerator
    print('Accelerator device:', Accelerator().device)
else:
    print('accelerate 未安装：pip install accelerate')
if has('transformers'):
    import transformers
    print('transformers', transformers.__version__)
else:
    print('transformers 未安装：pip install transformers')
print('trl:', 'installed' if has('trl') else 'missing（GRPOTrainer/SFTTrainer 见文档）')


## 思考题

1. 检查点为什么省显存？代价是什么？删掉 `use_reentrant=False` 会有什么差别？
2. `.view` 报错的两个修复（`reshape` 与 `contiguous().view`）在显存拷贝上有何不同？
3. 为什么 bf16 通常不需要 `GradScaler`，而 fp16 需要？`autocast` 会改参数 dtype 吗？
4. 用 profiler 观察：在训练循环里插入 `.item()` 会引入什么开销？

## 预期输出
```
torch 2.x | cuda True/False
前向一致: True
梯度一致: True
transpose 后 is_contiguous: False
view 报错: view size is not compatible ...
device: cpu | 输出 dtype: torch.bfloat16
参数 dtype 仍为: torch.float32
（profiler 的算子耗时表）
backend: eager | 前向: (16, 1)
LoraConfig ok: r=8 alpha=16
```
